# Sleeper API Weekly Stats Ingestion

Pull weekly NFL stats and player information directly from Sleeper's public API and land in bronze/silver Delta tables.

**Features:**
- ✅ **100% Free** - No API key required
- ✅ **No Authentication** - Public endpoints
- ✅ **Real-time Data** - Player news, injuries, stats
- ✅ **Comprehensive** - 7,000+ players with detailed info
- ✅ **Trending Data** - Add/drop trends across leagues

**Resources:**
- API Docs: https://docs.sleeper.com
- Base URL: https://api.sleeper.app/v1

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Sleeper API configuration
BASE_URL = "https://api.sleeper.app/v1"

# Configuration
SEASON = 2024  # Sleeper uses 'nfl' type, year is the season
WEEK = 18

print(f"📅 Fetching Sleeper data for Week {WEEK}, Season {SEASON}")
print("\nAvailable Sleeper API endpoints:")
print("  - All NFL players (7,000+ players)")
print("  - Player stats by week")
print("  - Player trending (add/drop trends)")
print("  - Player news and injuries")
print("  - Projections (from Sleeper's models)")

In [0]:
# Fetch all NFL players from Sleeper
# This gives us player metadata, injury status, team info
print("Fetching all NFL players from Sleeper...")

try:
    response = requests.get(f"{BASE_URL}/players/nfl", timeout=30)
    response.raise_for_status()
    all_players = response.json()
    
    print(f"Fetched {len(all_players)} total NFL players from Sleeper")
    
    # Convert to list format for easier processing
    players_list = []
    for player_id, player_data in all_players.items():
        player_data['sleeper_id'] = player_id
        players_list.append(player_data)
    
    print(f"\nSample player data fields:")
    if players_list:
        sample_player = players_list[0]
        print(f"  Available fields: {', '.join(list(sample_player.keys())[:10])}...")
    
    # Store for later use
    sleeper_players = all_players
    
except Exception as e:
    print(f"Error fetching Sleeper players: {e}")
    sleeper_players = {}

In [0]:
# Fetch stats for a specific week
# Sleeper provides stats in a simple format
print(f"\nFetching Week {WEEK} stats from Sleeper...")

try:
    # Sleeper stats endpoint: /stats/nfl/{season_type}/{season}/{week}
    # season_type can be 'regular', 'pre', 'post'
    season_type = 'regular'
    
    response = requests.get(
        f"{BASE_URL}/stats/nfl/{season_type}/{SEASON}/{WEEK}",
        timeout=30
    )
    response.raise_for_status()
    weekly_stats = response.json()
    
    print(f"Fetched stats for {len(weekly_stats)} players")
    
    # Build rows combining player info and weekly stats
    rows = []
    for player_id, stats in weekly_stats.items():
        # Get player metadata
        player_info = sleeper_players.get(player_id, {})
        
        # Calculate fantasy points (PPR scoring)
        # Sleeper provides pts_ppr field
        fantasy_points = float(stats.get('pts_ppr', 0) or 0)
        
        # Combine player info and stats
        combined_stats = {
            'player_id': player_id,
            'player_name': player_info.get('full_name', 'Unknown'),
            'position': player_info.get('position', 'Unknown'),
            'team': player_info.get('team', 'FA'),
            'injury_status': player_info.get('injury_status'),
            'fantasy_points_ppr': fantasy_points,
            'stats': stats,
            'metadata': {
                'age': player_info.get('age'),
                'years_exp': player_info.get('years_exp'),
                'college': player_info.get('college'),
                'status': player_info.get('status')
            }
        }
        
        rows.append(
            Row(
                player_id=str(player_id),
                week=WEEK,
                season=SEASON,
                fantasy_points=fantasy_points,
                stats=json.dumps(combined_stats),
                source='sleeper'
            )
        )
    
    if rows:
        stats_df = spark.createDataFrame(rows)
        display(stats_df)
        print(f"\n✓ Created DataFrame with {len(rows)} player records")
    else:
        print("No stats data available for this week")
        
except Exception as e:
    print(f"Error fetching weekly stats: {e}")
    print("\nNote: Stats may not be available yet for future weeks/seasons")
    print("Trying alternative approach...")
    
    # Fallback: create empty DataFrame
    stats_df = None

In [0]:
# Optional: Fetch trending players (add/drop trends)
# This shows which players are hot in the fantasy community
print("\nFetching trending players...")

try:
    # Trending adds
    response_add = requests.get(
        f"{BASE_URL}/players/nfl/trending/add",
        timeout=10
    )
    trending_add = response_add.json() if response_add.status_code == 200 else []
    
    # Trending drops
    response_drop = requests.get(
        f"{BASE_URL}/players/nfl/trending/drop",
        timeout=10
    )
    trending_drop = response_drop.json() if response_drop.status_code == 200 else []
    
    print(f"Trending adds (last 24h): {len(trending_add)} players")
    print(f"Trending drops (last 24h): {len(trending_drop)} players")
    
    if trending_add:
        print("\nTop 5 trending adds:")
        for i, player in enumerate(trending_add[:5]):
            player_id = player.get('player_id')
            player_info = sleeper_players.get(player_id, {})
            count = player.get('count', 0)
            print(f"  {i+1}. {player_info.get('full_name', player_id)} (+{count} adds)")
    
except Exception as e:
    print(f"Error fetching trending data: {e}")

In [0]:
# Write to bronze table using MERGE
if 'stats_df' in locals() and stats_df is not None:
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    
    # Create temp view for merge
    bronze_df.createOrReplaceTempView("sleeper_bronze_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING sleeper_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from Sleeper into bronze_weekly_stats")
else:
    print("⚠ No data to write - please run data fetch cells first")

In [0]:
# Transform for silver
if 'bronze_df' in locals() and bronze_df is not None:
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.col("ingested_at"),
        )
        .dropDuplicates(["player_id", "week", "season"])
    )
    
    # Create temp view for merge
    silver_df.createOrReplaceTempView("sleeper_silver_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING sleeper_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
    """)
    
    print(f"✓ Merged {silver_df.count()} records from Sleeper into silver_weekly_stats")
else:
    print("⚠ No data to write - please run bronze write cell first")

In [0]:
%sql
-- Check Sleeper data in silver table
SELECT 
  player_id,
  week,
  season,
  fantasy_points,
  get_json_object(stats, '$.player_name') as player_name,
  get_json_object(stats, '$.position') as position,
  get_json_object(stats, '$.team') as team
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2024
ORDER BY fantasy_points DESC
LIMIT 25

## Sleeper API Key Features

### 🆓 Completely Free
- No API key required
- No rate limits (reasonable use)
- No authentication needed
- All data publicly accessible

### 📊 Available Data
1. **Player Stats** - Weekly stats for all NFL players
2. **Player Info** - 7,000+ players with metadata
3. **Injuries** - Real-time injury status
4. **Trending** - Add/drop trends across all Sleeper leagues
5. **Projections** - Sleeper's own projections
6. **News** - Player news and updates

### 🎯 Key Endpoints
```python
# All NFL players
GET https://api.sleeper.app/v1/players/nfl

# Weekly stats
GET https://api.sleeper.app/v1/stats/nfl/{season_type}/{season}/{week}

# Projections
GET https://api.sleeper.app/v1/projections/nfl/{season_type}/{season}/{week}

# Trending adds
GET https://api.sleeper.app/v1/players/nfl/trending/add

# Trending drops
GET https://api.sleeper.app/v1/players/nfl/trending/drop
```

### 💡 Use Cases
- **Player Discovery** - 7,000+ players with rich metadata
- **Injury Tracking** - Real-time injury status
- **Community Sentiment** - See what players are trending
- **Historical Stats** - Access past weeks/seasons
- **Cross-reference** - Different player IDs than other sources

### 🔗 Player ID Mapping
Sleeper uses their own player IDs. To map to other sources:
- Use player name + team + position matching
- Sleeper provides `sportradar_id`, `espn_id`, `yahoo_id` in player data
- Build a mapping table for cross-referencing

### Next Steps
1. Run the notebook to test
2. Adjust WEEK and SEASON as needed
3. Schedule for regular updates
4. Consider adding projections endpoint